In [41]:
pip install opencv-python



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
pip install tensorflow



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [43]:
pip install mediapipe 


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [44]:
# import libraries
import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp
from collections import deque
from math import sqrt
import threading
import time
import csv
import os
from dataclasses import dataclass
from typing import Optional

In [45]:
# mediapipe stuff
BaseOptions  = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
HandLandmarkerResult = mp.tasks.vision.HandLandmarkerResult
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
GestureRecognizerResult = mp.tasks.vision.GestureRecognizerResult
VisionRunningMode = mp.tasks.vision.RunningMode

# from mediapipe hands
LANDMARK_NAMES = [
    'wrist',
    'thumb_cmc', 'thumb_mcp', 'thumb_ip',    'thumb_tip',
    'index_mcp', 'index_pip', 'index_dip',   'index_tip',
    'middle_mcp','middle_pip','middle_dip',  'middle_tip',
    'ring_mcp',  'ring_pip',  'ring_dip',    'ring_tip',
    'pinky_mcp', 'pinky_pip', 'pinky_dip',   'pinky_tip',
]

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),
]

def draw_hand(frame, landmarks):
    h, w, _ = frame.shape
    pts = [(int(l.x * w), int(l.y * h)) for l in landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (0, 255, 0), 2)
    for p in pts:
        cv2.circle(frame, p, 4, (0, 0, 255), -1)

In [46]:
# gesture mapping to define which gestures are active during a task 
TASK_GESTURES = {
    'confirm':['confirm'],
    'cancel':['cancel'],
    'back':['back'],
    'scroll':['scroll'],
    'zoom':['zoom'],
    'all':['confirm', 'cancel', 'back', 'scroll', 'zoom'], # for testing
}

# Expected Gestures per task to calculate success/failure in terms of recognition 
TASK_EXPECTED = {
    'confirm': 'confirm',
    'cancel': 'cancel',
    'back': 'back',
    'scroll': 'scroll',
    'zoom': 'zoom',
    'all': None,
}

TRIALS_PER_PHASE = 5 # five trials per task per phase (user/system)

### Gesture Detectors

In [47]:
# back gesture 
class BackGestureDetector:
    def __init__(self, model_path='model/back_gesture_model.tflite'):
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()             # tflite requires memory allocation before inference
        self.input_details = self.interpreter.get_input_details()      # retrieves metadata abt the model input tensor 21 land x 2 cood
        self.output_details = self.interpreter.get_output_details()     # classes 0/1

    def detect(self, hand_landmarks):
        if not hand_landmarks or len(hand_landmarks) < 21: # check if input is valid, catches no hand detected or incomplete landmarks 
            return False, 0.0
        
        # create empty list to store model input features
        features = [] 
        
        # loop thru landmarks 
        for lm in hand_landmarks:
            features.extend([lm.x, lm.y]) # ignore z bc model trained using only 2d cood + other issues talk abt them in diss
            
        # set the input tensor for the model
        self.interpreter.set_tensor(
            self.input_details[0]['index'],
            np.array([features], dtype=np.float32)      # convert features to numpy array
        )
        
        self.interpreter.invoke()       # execute model
        prob = float(self.interpreter.get_tensor(self.output_details[0]['index'])[0][1])    # read model output, retrieve the prediction result for [not back][back]
        
        return prob > 0.72, prob # if back gesture detected 

In [48]:
# ZOOM GESTURE DETECTOR
# ML model to confirm the zoom gesture
# pinch distance to determine if they actually are zooming 

class ZoomGestureDetector:
    CONF_THRESHOLD = 0.72     # min ml confidence required to confirm the gesture 
    PINCH_START = 0.04     # pinch sensitivity, lower it for higher sensitivity 
    PINCH_END = 0.08     # (dist threshold where pinch is considered release
    DELTA_THRESHOLD = 0.005    # min movement needed to register as a zoom movement
    SMOOTHING_WINDOW = 5        # num of frames used to avf ml conf (noise reduction)

    def __init__(self, model_path='model/zoom_gesture_model.tflite'):
        self.interpreter = tf.lite.Interpreter(model_path = model_path)
        self.interpreter.allocate_tensors()                         
        self.input_details  = self.interpreter.get_input_details()  # store input details
        self.output_details = self.interpreter.get_output_details() # store output details
        self.conf_history  = deque(maxlen = self.SMOOTHING_WINDOW)  # creates a fixed-size Queue storing recent confidence scores
        self.pinch_active  = False    # track whether a pinch gesture is currently happening 
        self.pinch_reference = None     # stores initial pinch distance
        
    # Run the ml model same as back
    def _run_model(self, hand_landmarks):
        
        features = []
        
        for lm in hand_landmarks:
            features.extend([lm.x, lm.y])
            
        self.interpreter.set_tensor(
            self.input_details[0]['index'],
            np.array([features], dtype=np.float32)
        )
        
        self.interpreter.invoke()
        return float(self.interpreter.get_tensor(self.output_details[0]['index'])[0][1])

    # main detection func
    def detect(self, hand_landmarks):
        if not hand_landmarks or len(hand_landmarks) < 21:
            self.pinch_active = False
            return None, 0.0

        self.conf_history.append(self._run_model(hand_landmarks)) # run model and store confidence score
        zoom_conf = float(np.mean(self.conf_history))   #avg confidence

        # calc distance between thumb & index
        thumb = hand_landmarks[4]
        index = hand_landmarks[8]
        
        # compute pinch distance using euclidean distance btw thumb n index
        distance = sqrt((thumb.x - index.x)**2 + (thumb.y - index.y)**2)

        # detect pinch start = if fingers close enough, start pinch
        if not self.pinch_active and distance < self.PINCH_START:
            self.pinch_active = True          # pinch active
            self.pinch_reference = distance      # store starting distance
            return None, 0.0                     # dont detect zoom yet, just 4 registering pinch start

        # handle active pinch 
        elif self.pinch_active:
            if distance > self.PINCH_END: # if fingers move too far apart
                self.pinch_active = False # end pinch gesture
                return None, 0.0      # reset detection
            
            if zoom_conf > self.CONF_THRESHOLD:   # ensure hand pose actually looks like a zoom gesture
                delta = distance - self.pinch_reference # calc difference from starting pinch distance
                
                if delta > self.DELTA_THRESHOLD:  # if fingers moved FURTHER APART
                    return 'zoom', zoom_conf
                if delta < -self.DELTA_THRESHOLD: #  if fingers move closer tgt
                    return 'zoom', zoom_conf

        return None, 0.0 # IF NO ZOOM

    # reset detector start, used when tassk changes/gesture interrupted
    def reset(self):
        self.pinch_active    = False 
        self.pinch_reference = None 
        self.conf_history.clear()   

In [49]:
# scroll detection
class DynamicGestureDetector:
    def __init__(self, window_size=15):
        self.hand_history = deque(maxlen=window_size)

    def add_frame(self, hand_landmarks):
        if hand_landmarks:
            self.hand_history.append(hand_landmarks)

    def detect(self):
        if len(self.hand_history) < 10:
            return None, 0.0
        y_positions = [hand[0].y for hand in self.hand_history]
        velocities  = np.diff(y_positions)
        max_vel     = np.max(np.abs(velocities))
        total_dist  = abs(y_positions[-1] - y_positions[0])
        if max_vel > 0.015 and total_dist > 0.08:
            mean_vel    = np.mean(velocities)
            consistency = np.sum(np.sign(velocities) == np.sign(mean_vel)) / len(velocities)
            if consistency > 0.72:
                return 'scroll', min(max_vel / 0.03, 1.0) * consistency
        return None, 0.0

    def reset(self):
        self.hand_history.clear()


#### TASK FILTERED 
build the main gesture recognition system that combines: mp built in gestures + my custom tflite models + scroll

In [50]:
# LIVE STREAM to show participants 
class LiveStreamStore:
    def __init__(self):
        self._lock = threading.Lock() # creates a thread lock 
        self.hand_landmarks = None  # stores the latest detected hand landmarks
        self.mp_gestures = None  # stores the latest MP gesture recognition result

    # hand callback
    def update_hand(self, result: HandLandmarkerResult, *_): 
        with self._lock: 
            self.hand_landmarks = result.hand_landmarks[0] if result.hand_landmarks else None

    def update_gesture(self, result: GestureRecognizerResult, *_): 
        with self._lock:
            self.mp_gestures = result

    def read(self): 
        with self._lock:
            return self.hand_landmarks, self.mp_gestures
        
# combined mediapipe gestures, my custom TFLite gestures, and the scroll
    
class TaskFilteredGestureRecognizer: # main gesture recognition system
    def __init__(self, current_task='all', mode='live_stream'):
        self.mode = mode 
        self.set_task(current_task) 

        if mode == 'live_stream':
            self._store = LiveStreamStore() 
            
            # create hand landmarker from MP
            self._hand_landmarker = HandLandmarker.create_from_options(
                HandLandmarkerOptions(
                    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
                    running_mode=VisionRunningMode.LIVE_STREAM,
                    num_hands=1,
                    result_callback=self._store.update_hand,
                )
            )
            
            # create mediapipe's built-in gesture recogniser 
            self._mp_recognizer = GestureRecognizer.create_from_options(
                GestureRecognizerOptions(
                    base_options=BaseOptions(model_asset_path='gesture_recognizer.task'),
                    running_mode=VisionRunningMode.LIVE_STREAM,
                    num_hands=1,
                    result_callback=self._store.update_gesture,
                )
            )
        
        # video mode (done for testing, but can be used to detect landmarks in a video)
        else:
            self._store = None
            self._hand_landmarker = HandLandmarker.create_from_options(
                HandLandmarkerOptions(
                    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
                    running_mode=VisionRunningMode.VIDEO,
                    num_hands=2,
                )
            )
            self._mp_recognizer = GestureRecognizer.create_from_options(
                GestureRecognizerOptions(
                    base_options=BaseOptions(model_asset_path='gesture_recognizer.task'),
                    running_mode=VisionRunningMode.VIDEO,
                    num_hands=2,
                )
            )
        
        
        # initialise the gesture models
        self.back_detector    = BackGestureDetector()
        self.zoom_detector    = ZoomGestureDetector()
        self.dynamic_detector = DynamicGestureDetector()
        self.cooldown         = 0


    # TASK SWITCHING
    def set_task(self, task_name):
        
        if task_name not in TASK_GESTURES: 
            task_name = 'all'
            
        self.current_task    = task_name              
        self.active_gestures = TASK_GESTURES[task_name] 
        
        
    # check if gesture is active
    def _active(self, name):
        return name in self.active_gestures

    
    # main detection pipeline
    def _process(self, hand_landmarks, mp_results):
        if self.cooldown > 0:
            self.cooldown -= 1

        # confirm/cancel
        if mp_results and mp_results.gestures:
            
            g = mp_results.gestures[0][0]
            
            if g.category_name == 'Thumb_Up' and g.score > 0.72 and self._active('confirm'):
                return 'confirm', g.score, 'static'
            
            if g.category_name == 'Thumb_Down' and g.score > 0.72 and self._active('cancel'):
                return 'cancel', g.score, 'static' 

        # back model
        if hand_landmarks:
            is_back, back_conf = self.back_detector.detect(hand_landmarks)
            
            if is_back and self._active('back'):
                return 'back', back_conf, 'static' 

        # 3)zoom model
        if hand_landmarks and self._active('zoom'):
            zoom_label, zoom_conf = self.zoom_detector.detect(hand_landmarks)
            
            if zoom_label:
                return zoom_label, zoom_conf, 'zoom'
        else:
            self.zoom_detector.reset()

        # 4) scroll
        if hand_landmarks:
            self.dynamic_detector.add_frame(hand_landmarks) # adds frame to motion tracker
        
        open_palm_active = False
        
        if mp_results and mp_results.gestures:
            g = mp_results.gestures[0][0]
            
            if g.category_name == "Open_Palm" and g.score>0.72:
                open_palm_active = True
            
            if self.cooldown == 0 and open_palm_active:
                scroll_label, scroll_conf = self.dynamic_detector.detect()
                
                if scroll_label and scroll_conf > 0.65 and self._active('scroll'): # runs scroll detection
                    self.cooldown = 20                                          
                    self.dynamic_detector.reset()                                  
                    return scroll_label, scroll_conf, 'dynamic'                    
        return None, 0.0, 'none'

    # live recognition model
    # adapted from: https://mediapipe.readthedocs.io/en/latest/solutions/hands.html 
    def recognize_live(self, frame_bgr, timestamp_ms: int):
        
        assert self.mode == 'live_stream'
        
        #converts to opencv format
        rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        
        #converts frame to MP image format
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        
        # runs hand detection + gesture recogniton asynchronously 
        self._hand_landmarker.detect_async(mp_img, timestamp_ms)
        self._mp_recognizer.recognize_async(mp_img, timestamp_ms)
        
        # retrieve latest callback results
        hand_landmarks, mp_results = self._store.read()
        
        # runs the gesture detection pipleine
        return self._process(hand_landmarks, mp_results)
    
    def recognize(self, hand_landmarks, mp_image, timestamp_ms: int):
        assert self.mode == 'video', "Call recognize_live() for LIVE_STREAM mode."
        mp_results = self._mp_recognizer.recognize_for_video(mp_image, timestamp_ms)
        return self._process(hand_landmarks, mp_results)
    
    # cleanup
    def close(self):
        self._hand_landmarker.close() 
        self._mp_recognizer.close()



## metrics for quantitative analysis

In [51]:
@dataclass 

class AttemptRecord:
    attempt_id: str
    participant_id: str
    task: str
    phase: str # 'user' or 'system'
    attempt_number: int # 1-5 within this task+phase change attempt to trial
    gesture: str # the gesture detected by the system
    expected: str # the gesture the participant was supposed to perform
    success: bool # whether the detection matched the expected gesture
    confidence: float # confidence score from recognition model
    reaction_time_ms: float
    frame_count: int # how many video frames were processed during the attempt (to analyse system performance) (did not enf up using bc not v sig)
    timestamp: str


class MetricsRecorder:
    CSV_FIELDS = [
        'attempt_id', 'participant_id', 'task', 'phase', 'attempt_number',
        'gesture', 'expected', 'success', 'confidence',
        'reaction_time_ms', 'frame_count', 'timestamp',
    ]

    def __init__(self, participant_id: str = 'P00', output_dir: str = 'results'):
        self.participant_id  = participant_id
        self.output_dir      = output_dir
        os.makedirs(output_dir, exist_ok=True)

        self._attempts: list[AttemptRecord] = []     
        self._attempt_start: Optional[float] = None    
        self._current_task: Optional[str] = None    
        self._current_expect: Optional[str] = None    
        self._current_phase: str = 'user'  
        self._attempt_recorded = False  
        self._attempt_counter = 0
        
        # track attempt num per combo
        self._phase_counters: dict = {}

        # create csv file
        self._csv_path = os.path.join(output_dir, f'{participant_id}_attempts.csv')
        
        with open(self._csv_path, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=self.CSV_FIELDS).writeheader()

    # phase control to change experiment phase
    def set_phase(self, phase: str):
        assert phase in ('user', 'system') 
        self._current_phase = phase
        

    # start a new gesture trial 
    def start_attempt(self, task: str, expected: str) -> str:
        key = (task, self._current_phase)
        
        # phase trial counter
        self._phase_counters[key] = self._phase_counters.get(key, 0) + 1

        self._attempt_counter += 1
        self._current_task = task
        self._current_expect = expected
        self._attempt_start = time.perf_counter()
        self._attempt_recorded = False
    
        
        attempt_id = f"{self.participant_id}_A{self._attempt_counter:03d}"
        
        num = self._phase_counters[key]
        
        print(f"{attempt_id}  [{self._current_phase.upper()} #{num}/{TRIALS_PER_PHASE}]  "
                f"task={task}  expected={expected}")
        
        return attempt_id

    def record(self, gesture: str, confidence: float, frame_count: int = 0):        
        if self._attempt_start is None or self._attempt_recorded:
            return
        
        rt = (time.perf_counter() - self._attempt_start) * 1000
        self._write(gesture, confidence, gesture == self._current_expect, rt, frame_count)
        
    def record_timeout(self, frame_count: int = 0):
        if self._attempt_start is None or self._attempt_recorded:
            return
        
        rt = (time.perf_counter() - self._attempt_start) * 1000
        self._write('none', 0.0, False, rt, frame_count)

    def _write(self, gesture, confidence, success, reaction_ms, frame_count):
        self._attempt_recorded = True
        
        key = (self._current_task, self._current_phase)
        confidence = float(confidence)
        rec = AttemptRecord(
            attempt_id = f"{self.participant_id}_{self._attempt_counter:02d}",
            participant_id = self.participant_id,
            task = self._current_task,
            phase = self._current_phase,
            attempt_number = self._phase_counters.get(key, 1),
            gesture = gesture,
            expected = self._current_expect,
            success = success,
            confidence = round(confidence, 4),
            reaction_time_ms = round(reaction_ms, 1),
            frame_count = frame_count,
            timestamp = time.strftime('%Y-%m-%dT%H:%M:%S'),
        )
        self._attempts.append(rec)
        with open(self._csv_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self.CSV_FIELDS).writerow(rec.__dict__)

        key  = (self._current_task, self._current_phase)
        num  = self._phase_counters.get(key, 1)
        print(f"  {icon}  [{self._current_phase.upper()} #{num}]  "
                f"task={rec.task}  detected={gesture}  "
                f"conf={confidence:.2f}  rt={reaction_ms:.0f}ms  frames={frame_count}")

    # summary
    def current_attempt_number(self) -> int:
        key = (self._current_task, self._current_phase)
        return self._phase_counters.get(key, 0)

    def summary(self) -> dict:
        if not self._attempts:
            return {}
        
        total     = len(self._attempts)
        successes = [a for a in self._attempts if a.success]
        rts       = [a.reaction_time_ms for a in self._attempts]
        confs     = [a.confidence for a in self._attempts if a.confidence > 0]
        
        stats = {
            'participant_id': self.participant_id,
            'total_attempts': total,
            'successes': len(successes),
            'failures': total - len(successes),
            'accuracy_%': round(len(successes) / total * 100, 1),
            'error_rate_%': round((total - len(successes)) / total * 100, 1),
            'mean_reaction_ms': round(float(np.mean(rts)), 1),
            'median_reaction_ms': round(float(np.median(rts)), 1),
            'std_reaction_ms': round(float(np.std(rts)), 1),
            'mean_confidence': round(float(np.mean(confs)), 4) if confs else 0.0,
            'per_task': {},
        }
        
        for task in TASK_GESTURES:
            for phase in ('user', 'system'):
                ta = [a for a in self._attempts if a.task == task and a.phase == phase]
                if not ta:
                    continue
                ts  = [a for a in ta if a.success]
                trt = [a.reaction_time_ms for a in ta]
                stats['per_task'][f'{task}/{phase}'] = {
                    'attempts':     len(ta),
                    'accuracy_%':   round(len(ts) / len(ta) * 100, 1),
                    'error_rate_%': round((1 - len(ts) / len(ta)) * 100, 1),
                    'mean_rt_ms':   round(float(np.mean(trt)), 1),
                }
        return stats

    def save_summary(self):
        stats = self.summary()
        if not stats:
            print("No attempts recorded.")
            return

        path = os.path.join(self.output_dir, f"{self.participant_id}_summary.csv")
        rows = [
            ['metric', 'value'],
            ['participant', stats['participant_id']],
            ['total_attempts', stats['total_attempts']],
            ['successes', stats['successes']],
            ['failures', stats['failures']],
            ['accuracy_%', stats['accuracy_%']],
            ['error_rate_%', stats['error_rate_%']],
            ['mean_reaction_ms', stats['mean_reaction_ms']],
            ['median_reaction_ms', stats['median_reaction_ms']],
            ['std_reaction_ms', stats['std_reaction_ms']],
            ['mean_confidence', stats['mean_confidence']],
            [], ['--- per task/phase ---', ''],
        ]
        for key, t in stats['per_task'].items():
            rows += [
                [f'{key}_attempts',     t['attempts']],
                [f'{key}_accuracy_%',   t['accuracy_%']],
                [f'{key}_error_rate_%', t['error_rate_%']],
                [f'{key}_mean_rt_ms',   t['mean_rt_ms']],
            ]
        with open(path, 'w', newline='') as f:
            csv.writer(f).writerows(rows)

        print(f"Saved: {path}")



In [52]:
# record hand landmark frames during a gesture attempt did not end up using in diss bc not significant
class LandmarkRecorder:
    
    #store experiment 
    META_COLS = [
        'attempt_id', 'participant_id', 'task', 'phase', 'attempt_number',
        'success', 'frame_index', 'timestamp_ms', 'pinch_distance', 'wrist_x', 'wrist_y',
    ]
    
    LM_COLS = []
    
    for _i in range(21):
        LM_COLS += [f'lm_{_i:02d}_x', f'lm_{_i:02d}_y', f'lm_{_i:02d}_z']

    def __init__(self, participant_id: str, output_dir: str = 'results'):
        self.participant_id  = participant_id
        self._path = os.path.join(output_dir, f'{participant_id}_landmarks.csv')
        self._buffer: list[dict] = []
        self._attempt_id: Optional[str] = None
        self._current_task: Optional[str] = None
        self._current_phase: str = 'user'
        self._attempt_number: int = 0
        self._frame_idx = 0
        self._recording = False

        os.makedirs(output_dir, exist_ok=True)
        with open(self._path, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=self.META_COLS + self.LM_COLS).writeheader()

    def start(self, attempt_id: str, task: str, phase: str, attempt_number: int):
        self._attempt_id = attempt_id
        self._current_task = task
        self._current_phase = phase
        self._attempt_number = attempt_number
        self._buffer.clear()
        self._frame_idx = 0
        self._recording = True

    def add_frame(self, landmarks, timestamp_ms: int):
        if not self._recording or landmarks is None:
            return
        thumb = landmarks[4]
        index = landmarks[8]
        wrist = landmarks[0]
        
        row = {
            'attempt_id': self._attempt_id,
            'participant_id': self.participant_id,
            'task': self._current_task,
            'phase': self._current_phase,
            'attempt_number': self._attempt_number,
            'success': None,   
            'frame_index': self._frame_idx,
            'timestamp_ms': timestamp_ms,
            'pinch_distance': round(sqrt((thumb.x-index.x)**2 + (thumb.y-index.y)**2), 6),
            'wrist_x': round(wrist.x, 6),
            'wrist_y': round(wrist.y, 6),
        }
        
        for i, lm in enumerate(landmarks):
            row[f'lm_{i:02d}_x'] =round(lm.x, 6)
            row[f'lm_{i:02d}_y'] =round(lm.y, 6)
            row[f'lm_{i:02d}_z'] = round(lm.z, 6)
        self._buffer.append(row)
        self._frame_idx += 1

    def flush(self, success: bool):
        if not self._recording:
            return
        self._recording = False
        for row in self._buffer:
            row['success'] = success
        if self._buffer:
            with open(self._path, 'a', newline='') as f:
                csv.DictWriter(f, fieldnames=self.META_COLS + self.LM_COLS).writerows(self._buffer)
        print(f"{len(self._buffer)} landmark frames flushed  (success={success})")
        self._buffer.clear()

    def stop_without_flush(self):
        self._recording = False
        self._buffer.clear()

    @property
    def frame_count(self) -> int:
        return self._frame_idx



##  main camera loop

In [53]:
def run(
    task:              str   = 'confirm',
    camera_index:      int   = 0,
    mode:              str   = 'live_stream',
    participant_id:    str   = 'P00',
    attempt_timeout_s: float = 10.0,
):

    recorder   = MetricsRecorder(participant_id = participant_id, output_dir='results')
    lm_rec     = LandmarkRecorder(participant_id = participant_id, output_dir='results')
    recognizer = TaskFilteredGestureRecognizer(current_task=task, mode=mode)

    cap        = cv2.VideoCapture(camera_index)
    start_time = time.time()

    last_gesture   = ''
    g_type         = 'none'
    gesture_timer  = 0
    DISPLAY_FRAMES = 20

    attempt_active    = False
    attempt_start_t   = None
    current_attempt_id = None

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame        = cv2.flip(frame, 1)
            timestamp_ms = int((time.time() - start_time) * 1000)

            #recognition
            if mode == 'live_stream':
                gesture, confidence, g_type = recognizer.recognize_live(frame, timestamp_ms)
                hand_landmarks, _ = recognizer._store.read() # retrieve hand landmarks
            else: # for video 
                rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                hand_results   = recognizer._hand_landmarker.detect_for_video(mp_img, timestamp_ms)
                hand_landmarks = hand_results.hand_landmarks[0] if hand_results.hand_landmarks else None
                gesture, confidence, g_type = recognizer.recognize(hand_landmarks, mp_img, timestamp_ms)

            if hand_landmarks:
                draw_hand(frame, hand_landmarks)

            if attempt_active:
                lm_rec.add_frame(hand_landmarks, timestamp_ms)

            # resolve attempt on detection
            if attempt_active and gesture:
                fc      = lm_rec.frame_count
                success = (gesture == TASK_EXPECTED.get(recognizer.current_task))
                
                recorder.record(gesture=gesture, confidence=confidence, frame_count=fc)
                lm_rec.flush(success=success)
                attempt_active = False
                last_gesture   = gesture
                gesture_timer  = DISPLAY_FRAMES

            #timeout
            if attempt_active and attempt_start_t:
                if (time.time() - attempt_start_t) > attempt_timeout_s:
                    fc = lm_rec.frame_count
                    recorder.record_timeout(frame_count=fc) # save metrics
                    lm_rec.flush(success=False)
                    attempt_active = False
                    last_gesture   = 'TIMEOUT'
                    gesture_timer  = DISPLAY_FRAMES

            elif gesture and not attempt_active:
                last_gesture  = gesture
                gesture_timer = DISPLAY_FRAMES

            # header display
            h, w = frame.shape[:2]
            cv2.rectangle(frame, (0, 0), (w, 100), (0, 0, 0), -1)

            attempt_num = recorder.current_attempt_number()
            def current_attempt_number(self):
                key = (self._current_task, self._current_phase)
                return self._phase_counters.get(key, 0)

            cv2.putText(frame,
                f"TASK: {recognizer.current_task.upper()}"
                f"  |  PHASE: {recorder._current_phase.upper()}"
                f"  |  ATTEMPT: {attempt_num}/{TRIALS_PER_PHASE}",
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (100, 200, 100), 2)

            cv2.putText(frame,
                f"Active gesture(s): {', '.join(recognizer.active_gestures)}",
                (10, 54), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

            if attempt_active and attempt_start_t:
                remaining  = max(0, attempt_timeout_s - (time.time() - attempt_start_t))
                frames_buf = lm_rec.frame_count
                cv2.putText(frame,
                    f"REC  {remaining:.1f}s remaining  |  {frames_buf} frames buffered",
                    (10, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)
            else:
                stats = recorder.summary()
                if stats:
                    cv2.putText(frame,
                        f"Acc: {stats['accuracy_%']}%  "
                        f"Err: {stats['error_rate_%']}%  "
                        f"Mean RT: {stats['mean_reaction_ms']}ms  "
                        f"n={stats['total_attempts']}",
                        (10, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (180, 255, 180), 1)
                else:
                    cv2.putText(frame, "Press SPACE to start attempt",
                        (10, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1)

            if gesture_timer > 0:
                color = (0, 200, 255) if last_gesture == 'TIMEOUT' else \
                        (0, 255, 165) if g_type == 'zoom'    else \
                        (255, 165, 0) if g_type == 'dynamic' else \
                        (0, 255, 0)
                cv2.putText(frame, last_gesture.upper(),
                    (10, 125), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3)
                gesture_timer -= 1
            
            cv2.imshow("Gesture User Study", frame)

            # key handling
            key = cv2.waitKey(1) & 0xFF

            def cancel_attempt():
                nonlocal attempt_active
                if attempt_active:
                    lm_rec.stop_without_flush()
                    attempt_active = False

            if key == ord('1'):
                cancel_attempt(); recognizer.set_task('confirm')
            elif key == ord('2'):
                cancel_attempt(); recognizer.set_task('cancel')
            elif key == ord('3'):
                cancel_attempt(); recognizer.set_task('back')
            elif key == ord('4'):
                cancel_attempt(); recognizer.set_task('scroll')
            elif key == ord('5'):
                cancel_attempt(); recognizer.set_task('zoom')
            elif key in (ord('a'), ord('A')):
                cancel_attempt(); recognizer.set_task('all')

            elif key in (ord('u'), ord('U')):
                cancel_attempt(); recorder.set_phase('user')
            elif key in (ord('s'), ord('S')):
                cancel_attempt(); recorder.set_phase('system')

            elif key == ord(' '):
                expected = TASK_EXPECTED.get(recognizer.current_task)
                current_attempt_id = recorder.start_attempt(
                    task = recognizer.current_task,
                    expected = expected or 'any',
                )
                key_combo  = (recognizer.current_task, recorder._current_phase)
                attempt_num = recorder._phase_counters.get(key_combo, 1)
                lm_rec.start(
                    attempt_id = current_attempt_id,
                    task = recognizer.current_task,
                    phase = recorder._current_phase,
                    attempt_number = attempt_num,
                )
                
                attempt_active = True
                attempt_start_t = time.time()

            elif key in (ord('t'), ord('T')):
                if attempt_active:
                    fc = lm_rec.frame_count
                    recorder.record_timeout(frame_count=fc)
                    lm_rec.flush(success=False)
                    attempt_active = False
                    last_gesture = 'TIMEOUT'
                    gesture_timer = DISPLAY_FRAMES

            elif key in (ord('m'), ord('M')):
                recorder.save_summary()

            elif key == 27:  # ESC to end + record stats
                break

    finally:
        if attempt_active:
            lm_rec.stop_without_flush()
        cap.release()
        recognizer.close()
        cv2.destroyAllWindows()
        recorder.save_summary()


In [54]:
if __name__ == '__main__':
    run(
        task              = 'confirm', #inital task 
        camera_index      = 0, #opencv, change according to your main camera (0 for primary, 1 for secondary)
        mode              = 'live_stream', #change to video if want to analyse videos
        participant_id    = 'Pxx', #participant number
        attempt_timeout_s = 5.0, #timeout can be changed
    )

I0000 00:00:1777651699.282505 17746350 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M1
W0000 00:00:1777651699.287360 17746353 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777651699.291933 17746353 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1777651699.295466 17746359 hand_gesture_recognizer_graph.cc:257] Custom gesture classifier is not defined.
I0000 00:00:1777651699.299674 17746359 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M1
W0000 00:00:1777651699.306229 17746367 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777651699.310889 17746364 inference_feedback_manager.cc:121] Feedback manager requires a mode

No attempts recorded.
